# **02_input_validation.ipynb**

**Programmers:**
* Albonia, Jade Lorenz M.
* Caspe, Mark Vincent G.
* Rivera, Rei Djemf M.
* Velante, Kamilah Kaye M.
* Villegas, Jedidiah S.

**Date Written:** September 2025

**Date Revised:** November 2025

---

### **System Context**
This notebook operates as the **Safety Layer (Gatekeeper)** for the A-EYE mobile application. Unlike the main diagnostic model, this lightweight classifier is deployed *upstream* to filter user inputs. It is executed on Kaggle to leverage diverse "Outlier" datasets (random objects, blurry photos, non-eye images) that are not present in the primary clinical dataset.

### **Purpose**
To train a **Binary Classifier (Valid Eye vs. Invalid Object)** that ensures the main diagnostic model only processes valid, pupil-based images.
1.  **Garbage Rejection:** Automatically rejects non-medical images to prevent "Hallucinated Diagnoses."
2.  **Quality Control:** Establishes a strict confidence threshold ($\theta = 0.8$) to filter out low-quality or ambiguous inputs.

---

### **Technical Architecture (Data Structures & Algorithms)**

**1. Data Structures**
* **Validation Dataset (Binary):** A curated mix of:
    * **Class 0 (Valid):** Cropped, high-resolution cataract and healthy eye images.
    * **Class 1 (Invalid/Outlier):** A noise dataset containing random internet images, blurry textures, and unrelated medical imagery.

**2. Algorithms**
* **MobileNetV2 Backbone:** Utilizes a pre-trained CNN (weights=`IMAGENET1K_V1`) frozen at the lower layers to act as a robust feature extractor for generic object detection.
* **Sigmoid Activation:** The final layer outputs a scalar probability, representing the "Validity Score."

**3. Control Flow**
* **Training Pipeline:** Standard PyTorch training loop with Binary Cross Entropy (`BCEWithLogitsLoss`) and Adam optimization.
* **Verification Test:** A final `quick_predict` routine iterates through a "Trap Set" of known invalid images. It enforces the rule: `Status = VALID if Prob > 0.8 else INVALID`, validating the logic that will be implemented in the mobile application.

In [ ]:
# ===================================================================
# A-EYE VALIDATION MODEL - MULTI-SOURCE FUSION & TRAINING
# ===================================================================
# Objective: Train a robust binary classifier (VALID vs INVALID) using
# multiple dataset sources per category, then save as .pth and .ptl.
# ===================================================================

# -- 1. SETUP AND IMPORTS --
!pip install torch torchvision torcheval --quiet
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import torch, torchvision, shutil
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import torch.nn as nn
import torch.optim as optim
from torcheval.metrics import BinaryAccuracy
from glob import glob
from PIL import Image

# ===================================================================
# -- 2. DATA CONFIGURATION (MULTI-SOURCE) --
# ===================================================================

# A. POSITIVE CLASS SOURCES (VALID)
VALID_SOURCES = [
    "/kaggle/input/new-valid/VALID_DATASET",
    "/kaggle/input/cataract-vs-normal-eye-dataset/Cataract/Test/Cataract",
    "/kaggle/input/cataract-vs-normal-eye-dataset/Cataract/Train/Cataract",
    "/kaggle/input/d/hemooredaoo/cataract/cataract-image-dataset/processed_images/test/cataract",
    "/kaggle/input/d/hemooredaoo/cataract/cataract-image-dataset/processed_images/train/cataract",
    "/kaggle/input/cataract/Cataract",
    "/kaggle/input/cataract-classification-dataset",
    "/kaggle/input/diseased-data/eye/Cataract"
]

# B. NEGATIVE CLASS SOURCES (INVALID)
# 1. Healthy Eyes
INVALID_HEALTHY_SOURCES = [
    "/kaggle/input/cataract-vs-normal-eye-dataset/Cataract/Test/Normal",
    "/kaggle/input/cataract-vs-normal-eye-dataset/Cataract/Train/Normal",
    "/kaggle/input/d/hemooredaoo/cataract/cataract-image-dataset/processed_images/test/normal",
    "/kaggle/input/d/hemooredaoo/cataract/cataract-image-dataset/processed_images/train/normal",
    "/kaggle/input/cataract/Normal",
    "/kaggle/input/conjunctivitis-dataset/Dataset/Normal",
    "/kaggle/input/diseased-data/eye/Normal_Eye",
    "/kaggle/input/iris-of-eye-dataset",
    "/kaggle/input/iris-images"
]

# 2. Animals
INVALID_ANIMAL_SOURCES = [
    "/kaggle/input/animal-image-dataset-90-different-animals/animals/animals",
    "/kaggle/input/animal-faces",
    "/kaggle/input/standford-dogs",
    "/kaggle/input/the-oxfordiiit-pet-dataset"
]

# 3. Objects/Random
INVALID_OBJECT_SOURCES = [
    "/kaggle/input/caltech-101/caltech-101",
    "/kaggle/input/food-101/food-101/food-101/images"
]

# 4. Outliers (Other Disease Eyes - Critical for logic)
INVALID_OUTLIER_SOURCES = [
    "/kaggle/input/outlier/OUTLIERS",
    "/kaggle/input/conjunctivitis-dataset/Dataset/Effected",
    "/kaggle/input/diseased-data/eye/Pterygium"
]

# --- Setup the working directory ---
FINAL_DATASET_DIR = "/kaggle/working/final_dataset"
VALID_DIR = os.path.join(FINAL_DATASET_DIR, "valid")
INVALID_DIR = os.path.join(FINAL_DATASET_DIR, "invalid")

# Clean previous run to avoid duplicates if re-running cell
if os.path.exists(FINAL_DATASET_DIR):
    shutil.rmtree(FINAL_DATASET_DIR)
os.makedirs(VALID_DIR, exist_ok=True)
os.makedirs(INVALID_DIR, exist_ok=True)

# --- Helper Function to Copy from Multiple Sources ---
def copy_images_from_list(source_list, dest_dir, limit_per_source=2000, label_prefix=""):
    total_copied = 0
    for src_path in source_list:
        if not os.path.exists(src_path):
            print(f"❌ Warning: Path not found: {src_path}")
            continue
            
        print(f"   -> Processing source: {os.path.basename(src_path)}...")
        count = 0
        # Recursive search for images
        image_paths = glob(os.path.join(src_path, '**', '*.png'), recursive=True) + \
                      glob(os.path.join(src_path, '**', '*.jpg'), recursive=True) + \
                      glob(os.path.join(src_path, '**', '*.jpeg'), recursive=True)
        
        for img_p in image_paths:
            if count < limit_per_source:
                try:
                    # Create unique filename to prevent overwriting same-named files from diff sources
                    filename = os.path.basename(img_p)
                    unique_name = f"{label_prefix}_{count}_{filename}"
                    shutil.copy(img_p, os.path.join(dest_dir, unique_name))
                    count += 1
                    total_copied += 1
                except Exception:
                    continue
            else:
                break
    print(f"✅ Finished '{label_prefix}'. Total images: {total_copied}")

# --- Assemble the Dataset ---
print("--- Step 1: Assembling Data from Multiple Sources ---")
# Adjust limits as needed based on how much data you have
copy_images_from_list(VALID_SOURCES, VALID_DIR, limit_per_source=5000, label_prefix="Valid_Cataract")
copy_images_from_list(INVALID_HEALTHY_SOURCES, INVALID_DIR, limit_per_source=3000, label_prefix="Inv_Healthy")
copy_images_from_list(INVALID_ANIMAL_SOURCES, INVALID_DIR, limit_per_source=3000, label_prefix="Inv_Animal")
copy_images_from_list(INVALID_OBJECT_SOURCES, INVALID_DIR, limit_per_source=3000, label_prefix="Inv_Object")
copy_images_from_list(INVALID_OUTLIER_SOURCES, INVALID_DIR, limit_per_source=5000, label_prefix="Inv_Outlier")

# ===================================================================
# -- 3. TRAINING CONFIGURATION --
# ===================================================================
print("\n--- Step 2: Preparing DataLoaders ---")
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(25), 
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.4, hue=0.1), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(root=FINAL_DATASET_DIR, transform=transform)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"Classes: {full_dataset.class_to_idx}")
print(f"Training Samples: {len(train_dataset)} | Validation Samples: {len(val_dataset)}")

# Calculate Class Weight (Handling Imbalance)
num_valid = len(glob(os.path.join(VALID_DIR, "*")))
num_invalid = len(glob(os.path.join(INVALID_DIR, "*")))
if num_valid > 0:
    pos_weight = torch.tensor([num_invalid / num_valid])
else:
    pos_weight = torch.tensor([1.0]) # Fallback
print(f"Positive Class Weight: {pos_weight.item():.2f}")

# ===================================================================
# -- 4. MODEL SETUP (DISCRIMINATIVE FINE-TUNING) --
# ===================================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n--- Step 3: Setting up MobileNetV2 on {device.upper()} ---")

model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

# Freeze base layers
for param in model.parameters():
    param.requires_grad = False

# Unfreeze last 10 feature layers + classifier for deep fine-tuning
layers_to_tune = list(model.features.children())[-10:] + list(model.classifier.children())
for layer in layers_to_tune:
    for param in layer.parameters():
        param.requires_grad = True

# Modify Classifier for Binary Output
model.classifier[1] = nn.Sequential(nn.Linear(model.classifier[1].in_features, 1))
model = model.to(device)
pos_weight = pos_weight.to(device)

# Discriminative Learning Rates (Critical for High Accuracy)
param_groups = [
    {'params': model.features[-10:-5].parameters(), 'lr': 1e-6},
    {'params': model.features[-5:].parameters(), 'lr': 5e-6},
    {'params': model.classifier.parameters(), 'lr': 2e-5}
]
optimizer = optim.Adam(param_groups) 
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
accuracy_metric = BinaryAccuracy(device=device)

# ===================================================================
# -- 5. TRAINING LOOP --
# ===================================================================
num_epochs = 30
print(f"Starting training for {num_epochs} epochs...")

for epoch in range(num_epochs):
    model.train()
    for imgs, labels in train_loader:
        imgs, labels_for_loss = imgs.to(device), labels.float().unsqueeze(1).to(device)
        outputs = model(imgs)
        loss = criterion(outputs, labels_for_loss)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        accuracy_metric.update(torch.sigmoid(outputs).squeeze(), labels_for_loss.squeeze())
        
    train_acc = accuracy_metric.compute()
    accuracy_metric.reset()

    model.eval()
    with torch.no_grad():
        for imgs, labels_for_loss in val_loader:
            imgs, labels_for_loss = imgs.to(device), labels_for_loss.float().unsqueeze(1).to(device)
            outputs = model(imgs)
            accuracy_metric.update(torch.sigmoid(outputs).squeeze(), labels_for_loss.squeeze())
            
    val_acc = accuracy_metric.compute()
    accuracy_metric.reset()
    print(f"Epoch [{epoch+1}/{num_epochs}] | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

# ===================================================================
# -- 6. SAVING MODELS (.pth AND .ptl) --
# ===================================================================
print("\n--- Step 4: Saving Models ---")
model.eval() 

# 1. Save State Dict (.pth)
torch.save(model.state_dict(), "aeye_validation_model.pth")
print("✅ Saved .pth (Python) model.")

# 2. Save TorchScript Lite (.ptl)
model_cpu = model.cpu()
scripted_model = torch.jit.script(model_cpu)
# Optimize for mobile
scripted_model._save_for_lite_interpreter("aeye_validation_model.ptl")
print("✅ Saved .ptl (Mobile) model.")

# ===================================================================
# -- 7. TESTING (VERIFICATION) --
# ===================================================================
print("\n--- Step 5: Verification Test on Sample Data ---")

# Define simple prediction function for the check
def quick_predict(path, model_obj):
    try:
        img = Image.open(path).convert('RGB')
        tns = transform(img).unsqueeze(0)
        with torch.no_grad():
            out = model_obj(tns)
            prob = torch.sigmoid(out).item()
        pred = "VALID" if prob > 0.8 else "INVALID"
        return f"{os.path.basename(path)}: {prob:.4f} -> {pred}"
    except: return "Error"

# Update these paths to your actual TEST folders
TEST_DIRS = {
    "Cataract (Expect VALID)": "/kaggle/input/validation-test/TEST/test_cataract",
    "Outlier (Expect INVALID)": "/kaggle/input/validation-test/TEST/test_outlier"
}

for category, path in TEST_DIRS.items():
    print(f"\nChecking: {category}")
    if os.path.exists(path):
        images = glob(os.path.join(path, "*"))[:3]
        for img in images:
            print(quick_predict(img, model_cpu))
    else:
        print("Test path not found.")

print("\n✅ DONE! Download 'aeye_validation_model.ptl' for your app.")